# Stage 3 — DPO Preference Alignment with Unsloth

We align the SFT model toward **better** answers using a preference dataset of
`(prompt, chosen, rejected)` triples. DPO (Direct Preference Optimization) trains
the model to raise the likelihood of the *chosen* answer relative to the *rejected*
one — improving helpfulness, safety, tone, and reducing generic replies.

**Input:** `data/preference_dataset.jsonl` + `models/sft_adapter/`
**Output:** LoRA adapter saved to `models/dpo_adapter/`

In [ ]:
# --- Environment setup (Google Colab, T4/A100 GPU) -----------------------
# Unsloth makes TinyLlama fine-tuning fit comfortably in a free Colab T4.
%%capture
# Install Unsloth and let it pull compatible recent transformers/trl/peft.
# (These notebooks use the CURRENT trl API: SFTConfig + processing_class.)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --upgrade trl peft accelerate bitsandbytes datasets
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

In [ ]:
# --- Get the project code + datasets into Colab -------------------------
# Option A: clone your GitHub repo (recommended once you have published it)
#   !git clone https://github.com/<your-username>/customer-support-assistant.git
#   %cd customer-support-assistant
#
# Option B: mount Google Drive and point PROJECT at your uploaded folder
#   from google.colab import drive; drive.mount('/content/drive')
#   PROJECT = '/content/drive/MyDrive/customer-support-assistant'
#
# For a local machine, just run the notebook from the project root.
import os
PROJECT = os.environ.get("PROJECT", ".")
DATA = os.path.join(PROJECT, "data")
MODELS = os.path.join(PROJECT, "models")
os.makedirs(MODELS, exist_ok=True)
print("PROJECT:", os.path.abspath(PROJECT))

## 1. Load the SFT model (with Unsloth DPO patch)

In [ ]:
from unsloth import FastLanguageModel, PatchDPOTrainer
PatchDPOTrainer()   # must be called before building the DPOTrainer

import torch, os
MAX_SEQ_LENGTH = 2048
BASE_MODEL = "unsloth/tinyllama"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = True,
)

sft_dir = os.path.join(MODELS, "sft_adapter")
assert os.path.isdir(sft_dir), f"Run Stage 2 first - {sft_dir} not found"
model.load_adapter(sft_dir, adapter_name="default")

# Make the loaded SFT adapter trainable for DPO
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0.0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 2. Load and format the preference dataset

In [ ]:
from datasets import load_dataset

# DPO expects columns: prompt, chosen, rejected.
# We wrap the prompt in the SAME template used during SFT, and format the
# responses as the completion the model should produce.
PROMPT = "### Instruction:\n{q}\n\n### Response:\n"

def format_pref(ex):
    return {
        "prompt":   PROMPT.format(q=ex["prompt"]),
        "chosen":   ex["chosen"]   + tokenizer.eos_token,
        "rejected": ex["rejected"] + tokenizer.eos_token,
    }

pref_path = os.path.join(DATA, "preference_dataset.jsonl")
pref = load_dataset("json", data_files=pref_path, split="train").map(format_pref)
print("preference examples:", len(pref))
print(pref[0])

## 3. Configure and run DPO training

In [ ]:
from trl import DPOTrainer, DPOConfig

dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,           # Unsloth/PEFT uses the frozen base as implicit reference
    processing_class = tokenizer,   # was: tokenizer = tokenizer
    train_dataset = pref,
    args = DPOConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.1,
        num_train_epochs = 2,
        learning_rate = 5e-6,       # DPO uses a smaller LR than SFT
        beta = 0.1,                 # KL strength - how far from the SFT policy
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.0,
        lr_scheduler_type = "linear",
        seed = 3407,
        max_length = MAX_SEQ_LENGTH,
        max_prompt_length = 512,
        output_dir = "outputs_stage3",
        report_to = "none",
    ),
)
dpo_trainer.train()

## 4. Test the model AFTER DPO

In [ ]:
FastLanguageModel.for_inference(model)

def answer(q, max_new_tokens=160):
    text = f"### Instruction:\n{q}\n\n### Response:\n"
    inputs = tokenizer([text], return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         temperature=0.7, top_p=0.9, do_sample=True)
    return tokenizer.batch_decode(out, skip_special_tokens=True)[0].split("### Response:")[-1].strip()

for q in ["How can I cancel my order?",
          "My order says delivered but I did not receive it.",
          "Someone may have accessed my account."]:
    print("Q:", q); print("A:", answer(q)); print("-"*70)

## 5. Save the DPO-aligned adapter

In [ ]:
save_dir = os.path.join(MODELS, "dpo_adapter")
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print("Saved DPO-aligned adapter to:", save_dir)

# (Optional) export a merged 16-bit model or GGUF for local CPU inference:
# model.save_pretrained_merged(os.path.join(MODELS,"dpo_merged"), tokenizer, save_method="merged_16bit")
# model.save_pretrained_gguf(os.path.join(MODELS,"dpo_gguf"), tokenizer, quantization_method="q4_k_m")

### Stage 3 done
Fill in `reports/final_evaluation.md` comparing Base vs SFT vs DPO, then use
`src/inference.py` to chat with the final aligned model.